# Variational Autoencoders
_Turning a latent space into a probability distribution_

---

This lab builds a variational autoencoder and applies it to (i) digit generation and
(ii) anomaly detection, on the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) database,
the dataset of the **autoencoder lab**, which this one builds on.

The change with respect to that lab is that the encoder no longer produces a point of the latent
space, but a **distribution** over it. That is the whole difference, and everything else follows
from it: sampling that distribution is what makes generation possible, and comparing it to a
prior is what makes the latent space usable.

> This notebook runs on its own. The few things it borrows from the autoencoder lab are given
> again, as provided code, at the point where they are needed.

<br>
<div><img src="img/vae_mnist.png" width="600px" style="display:block; margin-left:auto; margin-right:auto;"/></div>

---

## Conventions used in this lab

This lab is written with `PyTorch`. Four conventions are used throughout, the same four as in
the autoencoder lab, and are worth stating once and for all.

**1. Two input layouts, and the `view` that goes from one to the other.** A `DataLoader` built
on MNIST returns images as `(N, 1, 28, 28)`. The networks of this lab are dense and take a
vector, so their inputs are flattened with `x.view(x.size(0), -1)` into `(N, 784)`; only the
convolutional variational autoencoder of the last section takes the images as they come.

**2. The decoder ends with a sigmoid, and the reconstruction term is the binary cross-entropy.**
As in the autoencoder lab, and contrary to the usual `PyTorch` practice of keeping raw scores
inside the model, the decoder outputs values in $[0, 1]$, like the pixels of the normalized
images. This follows the seminal article
[[Kingma & Welling, 2014]](https://arxiv.org/pdf/1312.6114.pdf) and keeps the reconstruction term
readable. The numerically stable alternative would be to output raw scores and use
`binary_cross_entropy_with_logits`; nothing in this lab depends on that choice.

**3. The test set plays the role of a validation set.** There is no third split here, and the
loss printed as `Val Loss` during training is computed on the test set. Nothing is ever
*selected* on it (no early stopping, no search over the hyperparameters), so the figures stay
honest; had we wanted to choose between several runs, a genuine validation set would have been
needed, as in the VisionCNN lab.

**4. Everything is sent to the same `device`.** The models and *all* the tensors given to them,
including the latent vectors drawn by hand to generate images. A tensor left on the CPU while
the model sits on the GPU is the most common error in this lab, and it does not show up on a
machine that has no GPU.

## Setting up the environment

In [ ]:
import math
import random

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset, ConcatDataset
from torchvision import datasets
from torchvision.transforms import v2
from torchinfo import summary

print("torch version:", torch.__version__)

In [ ]:
from tqdm import tqdm
#from tqdm.notebook import tqdm

from sklearn.manifold import TSNE
from scipy.stats import norm

In [ ]:
# All the tensors and models of this lab will be sent to this device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Pinning host memory only speeds up transfers towards an accelerator: without one,
# it does nothing, and recent versions of PyTorch warn about it at every DataLoader.
PIN_MEMORY = (device.type == "cuda")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## The [MNIST](https://en.wikipedia.org/wiki/MNIST_database) database

As seen earlier in the course, the stability of the algorithms is improved by normalizing the data.
In `PyTorch`, transformations are applied to the data at loading time: this is what we do here.

In [ ]:
# Transform: from a PIL image to a float tensor with values in [0, 1]
transform = v2.Compose([
    v2.ToImage(),                           # PIL image -> tensor, with an explicit channel dimension
    v2.ToDtype(torch.float32, scale=True),  # uint8 in [0, 255] -> float32 in [0, 1]
])

# Load datasets
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [ ]:
print(f"Train set: {len(train_dataset)} images of size {train_dataset.data.shape[1]} x {train_dataset.data.shape[2]}")
print(f"Test set:  {len(test_dataset)} images")

In order to train the networks more easily afterwards, we create a `DataLoader` to access the
data. In particular, we need to specify the size of the (future) training batches.

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 2,
    pin_memory = PIN_MEMORY)

test_loader = DataLoader(
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 2,
    pin_memory = PIN_MEMORY)

A batch can then be accessed with the command `next(iter(train_loader))`.

In [ ]:
images, labels = next(iter(train_loader))

print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

The following code displays sample images.

In [ ]:
n = 10

plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i+1)
    plt.imshow(images[i][0], cmap="gray")
    ax.grid(False)
    plt.axis("off")
plt.show()

## Two helpers from the autoencoder lab

The two functions below were written in the autoencoder lab. They are given again here, unchanged,
so that this notebook runs on its own: `check_device_model` puts a model on the right device, and
`plot_images` displays rows of images and is used by nearly every figure of this lab.

In [ ]:
def check_device_model(model, device=device):
    """Move `model` to `device`. Calling it on a model already there costs nothing."""
    return model.to(device)


# --- #

def plot_images(imgs, sz, titles, n, cmap="gray"):
    """Display several rows of images, one row per entry of `imgs`.

    imgs   : list of batches, each a tensor or an array of at least `n` images
    sz     : shape each row must be reshaped to, e.g. (28, 28)
    titles : title of each row, placed above its first image
    n      : number of images displayed per row
    """
    num_rows = len(imgs)
    plt.figure(figsize=(2*n, 2*num_rows))

    for row, images in enumerate(imgs):
        # Accept tensors as well as arrays: bring everything back to numpy on the CPU
        if torch.is_tensor(images):
            images = images.detach().cpu().numpy()

        for i in range(n):
            ax = plt.subplot(num_rows, n, row*n + i + 1)
            plt.imshow(images[i].reshape(sz[row]), cmap=cmap)
            ax.axis("off")

        plt.subplot(num_rows, n, row*n + 1).set_title(titles[row], fontsize=12)

    plt.tight_layout()
    plt.show()

## Building the variational autoencoder

### Encoder

First we build the **encoder**. It consists of:

1. a dense layer of `intermediate_dim = 512` neurons, with a $\texttt{ReLU}$ activation function;
2. two dense layers of `latent_dim = 2` neurons **above that same first layer**, with a linear
   activation function. These two layers produce the two variables `z_mean` and `z_log_var` of
   the latent space.

> The second output is the **logarithm of the variance**, not the variance itself, and not the
> standard deviation. A network output runs over the whole real line; taking its exponential
> gives a positive variance for free, with no constraint to enforce.

In [ ]:
# network parameters
input_dim = 28 * 28
intermediate_dim = 512
latent_dim = 2

##### <i style="color:teal">**Todo:** Write a `PyTorch` code implementing the encoder described above</i>

In [ ]:
### TO BE COMPLETED ###

class VAEEncoder(nn.Module):
    """Encoder of the VAE, before the reparametrization trick."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/vae/Encoder.py

#### Stochastic latent variable

We use the reparametrization trick to define the latent random variable $z$ conditionally on the
input image $x$, according to the normal distribution
$$ z\vert x \,\sim\, \mathcal{N}\big(\mu_z(x),\, \Sigma_z(x)\big) \,, \qquad
   \Sigma_z(x) \,=\, \mathrm{diag}\big(\sigma_{z,1}^2(x), \dots, \sigma_{z,d}^2(x)\big) \,, $$
where $d$ is the dimension of the latent space. The covariance matrix is diagonal: the
coordinates of $z$ are independent conditionally on $x$, which is what lets the network output
one variance per coordinate rather than a full matrix.

<br>
<div><img src="img/vae_3.svg" width="600px" style="display:block; margin-left:auto; margin-right:auto;"/></div>
<br>

The reparametrization trick consists in rewriting $z$ as
$$ z\vert x \,=\, \mu_z(x) + \sigma_z(x)\odot\varepsilon
   \qquad\text{with}\qquad \varepsilon\sim\mathcal{N}(0, I_d) \,, $$
where $\sigma_z(x)$ is the vector of the **standard deviations**, _i.e._ the square root of the
diagonal of $\Sigma_z(x)$, and $\odot$ the coordinate-wise product.

Written this way, the dependency between $z$ and $x$ is deterministic and _differentiable_, so
the gradient goes through it. All the randomness of $z$, at fixed $x$, is carried by
$\varepsilon$ alone, which depends on no parameter.

> **Notation.** From here on $\sigma_z$ always denotes the standard deviation and $\sigma_z^2$
> the variance. The network outputs $\log \sigma_z^2$, called `z_log_var` in the code, hence the
> `torch.exp(0.5 * log_var)` below, which is $\sigma_z$.

This leads us to modify the `VAEEncoder` class as follows.

In [ ]:
class VAEEncoder(nn.Module):
    """Encoder of the VAE, with the reparametrization trick.

    `forward` now returns three things: the sample z, and the two parameters of the
    distribution it was drawn from, which the loss needs.
    """

    def __init__(self, input_dim, intermediate_dim, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, intermediate_dim)
        self.fc_mean = nn.Linear(intermediate_dim, latent_dim)
        self.fc_log_var = nn.Linear(intermediate_dim, latent_dim)

    def reparameterize(self, mean, log_var):
        std = torch.exp(0.5 * log_var)   # log_var = log(sigma^2), so this is sigma
        eps = torch.randn_like(std)
        return mean + eps * std

    def forward(self, x):
        h = F.relu(self.fc1(x))
        z_mean = self.fc_mean(h)
        z_log_var = self.fc_log_var(h)
        z = self.reparameterize(z_mean, z_log_var)
        return z, z_mean, z_log_var


vae_encoder = VAEEncoder(input_dim, intermediate_dim, latent_dim)
summary(vae_encoder, input_size=(1, input_dim))

### Decoder

The decoder takes the vector $z$ as input, _i.e._ the sample of the latent distribution produced
by the encoder. It consists of two dense layers:

* `intermediate_dim = 512` neurons, $\texttt{ReLU}$ activation;
* `input_dim = 784` neurons, sigmoid activation.

##### <i style="color:teal">**Todo:** Build this decoder</i>

In [ ]:
### TO BE COMPLETED ###

class VAEDecoder(nn.Module):
    """Decoder of the VAE: from the latent vector back to an image of `input_dim` pixels."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/vae/Decoder.py

### Autoencoder

We can now combine the encoder and the decoder to define our variational autoencoder.

> Mind the order of the arguments when instantiating the decoder: its signature is
> `VAEDecoder(input_dim, intermediate_dim, latent_dim)`, in that order, exactly as for the encoder.

##### <i style="color:teal">**Todo:** Build this VAE</i>

In [ ]:
### TO BE COMPLETED ###

class VAE_mlp(nn.Module):
    """Variational autoencoder made of the encoder and the decoder written above."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/vae/VAE_mlp.py

#### Loss function

We now implement the loss function of the VAE, as described in the course:
$$ \mathcal{L}_{VAE} \,=\, \mathcal{L}(x,\hat{x}) \,+\, KL\big(\, q(z\vert x) \,\Vert\, p(z) \,\big) \,, $$
where $\mathcal{L}$ is a loss adapted to our problem between the original image $x$ and the
reconstructed image $\hat{x}$ (as in the autoencoder lab, we take the binary
cross-entropy), and $KL$ denotes the Kullback-Leibler divergence between the distribution
$q(z\vert x)$ produced by the encoder and the prior $p$ placed on the latent variable $z$.

This quantity is the opposite of the ELBO, the lower bound on the log-likelihood of the data:
minimizing $\mathcal{L}_{VAE}$ is maximizing that bound.

Given that $z\vert x \sim \mathcal{N}(\mu_z(x), \Sigma_z(x))$, we choose as prior for $z$ the
standard Gaussian distribution, $z\sim\mathcal{N}(0, I_d)$.

> **A caveat on the reconstruction term.** The binary cross-entropy is the log-likelihood of a
> Bernoulli distribution, which assumes $x_k \in \{0, 1\}$. The pixels of MNIST are grey levels
> in $[0, 1]$, so $\mathcal{L}(x, \hat{x})$ is not a log-likelihood here, and
> $\mathcal{L}_{VAE}$ is the opposite of the ELBO only up to a term we leave out. Two clean ways
> out: binarize $x$, or replace the Bernoulli by the *continuous Bernoulli*, whose normalizing
> constant depends on $\hat{x}$
> ([Loaiza-Ganem & Cunningham, NeurIPS 2019](https://arxiv.org/abs/1907.06845)). We keep the grey
> levels and the plain cross-entropy, as is customary, but the caveat comes back in the anomaly
> detection section, where the ELBO is used as a score.

<p style="color:teal">
<b>Proposition</b>: Consider two Gaussians $\, \mathcal{N}_d(\mu_p,\Sigma_p) \,$ and $\, \mathcal{N}_d(\mu_q,\Sigma_q)$. Then their Kullback-Leibler divergence is given by:
$$
KL\big(\, \mathcal{N}_d(\mu_q,\Sigma_q) \,\Vert\, \mathcal{N}_d(\mu_p,\Sigma_p) \big) \,=\, \frac12 \left[
\log\frac{\vert\Sigma_p\vert}{\vert\Sigma_q\vert} - d + \textrm{tr}\big(\Sigma_p^{-1}\Sigma_q \big) + \big(\mu_p -
\mu_q\big)^\top \Sigma_p^{-1}\big(\mu_p - \mu_q\big)\right] \,.
$$
</p>

<br>

In our context, $\, p = \mathcal{N}_d(0, I_d) \,$ and $\, q(\cdot\vert x) = \mathcal{N}_d(\mu_z(x), \Sigma_z(x)) \,$,
so $\Sigma_p = I_d$ and $\Sigma_q = \Sigma_z(x)$. The proposition gives
$$
KL\big(\, q(z\vert x) \,\Vert\, p(z) \,\big) \,=\, \frac12 \left[-\log\vert\Sigma_z(x)\vert - d + \text{tr} \big(\Sigma_z(x)\big) + \mu_z(x)^\top \mu_z(x)\right] \,,
$$
and, since $\Sigma_z(x)$ is diagonal with entries $\sigma_{z,j}^2(x)$,
$$
KL\big(\, q(z\vert x) \,\Vert\, p(z) \,\big) \,=\, \frac12 \, \sum_{j=1}^d \Big[ \sigma_{z,j}^2(x) + \mu_{z,j}^2(x) - 1 - \log\sigma_{z,j}^2(x) \Big] \,.
$$

The network gives us $\log\sigma_{z,j}^2(x)$ directly, as `z_log_var`: the last term is that
output, and the first is its exponential.

##### <i style="color:teal">**Todo:** Using this last expression, define the loss function of the VAE</i>

Beware of the reduction: the KL term above is a sum over the latent dimensions, and it is summed
over the batch as well. The reconstruction term must therefore be summed too, and not averaged,
for the two to be comparable. Hence the `reduction="sum"` of
`F.binary_cross_entropy`. The division by the number of images is done once, in the training
loop.

In [ ]:
### TO BE COMPLETED ###

def vae_loss(x, x_hat, mean, log_var):
    """Opposite of the ELBO, summed over the batch."""

    reconstruct_loss = ... ### TO BE COMPLETED ###
    kl = ...               ### TO BE COMPLETED ###

    return reconstruct_loss + kl

In [ ]:
# %load solutions/vae/vae_loss.py

### Training and results

In [ ]:
input_dim = 28 * 28
intermediate_dim = 512
latent_dim = 2

LEARNING_RATE = 1e-3
EPOCHS = 20

##### <i style="color:teal">**Todo:** Taking the loops of the autoencoder lab as inspiration, train the VAE</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/vae_mlp_train.py

We can now quickly check the performance of the model by displaying sample test images and their
reconstruction.

In [ ]:
def reconstruct_batch(model, loader, device=device):
    """Reconstruct one batch of `loader` and return the originals and the reconstructions."""
    model = check_device_model(model, device=device)
    model.eval()

    x, _ = next(iter(loader))
    x = x.view(x.size(0), -1).to(device)   # flatten
    with torch.no_grad():
        x_hat, _, _ = model(x)

    return x.view(-1, 1, 28, 28), x_hat.view(-1, 1, 28, 28)


# --- #
x, x_hat = reconstruct_batch(vae, test_loader)
plot_images(
    imgs = [x, x_hat],
    sz = [(28, 28), (28, 28)],
    titles = ['Original', 'Decoded'],
    n = 10
)

## Training visualization

In order to visualize the training, we would like to display how the reconstructions evolve
along the epochs.

In [ ]:
n = 10  # number of images to visualize
m = 10  # number of epochs to display

##### <i style="color:teal">**Todo:** Complete the following code to display the first $m$ training steps for $n$ images</i>

* each step, including the initialization and the real image, is represented on a single row;
* each column corresponds to one image, transformed step by step;
* in particular, the complete figure has $n$ columns and $m+2$ rows.

In [ ]:
### TO BE COMPLETED ###

# Init model and optimizer
vae_seq = VAE_mlp(input_dim, intermediate_dim, latent_dim).to(device)
optimizer = torch.optim.Adam(..., lr=LEARNING_RATE)  ### TO BE COMPLETED ###

# --- #
# Select test images for plotting
idx = torch.randint(0, len(test_dataset), (n,))

test_images = []
for i in idx:
    img, _ = test_dataset[i]
    test_images.append(img)
x_test_sample = torch.stack(test_images).view(n, -1).to(device)

imgs = [x_test_sample]
titles = ['Images']


# --- #
# Reconstruction before training (epoch 0)
vae_seq.eval()
with torch.no_grad():
    x_test_decoded = vae_seq(x_test_sample)[0].view(n, 28, 28)

imgs.append(x_test_decoded)
titles.append('Init')


# --- #
# Training loop
for j in range(m):
    print(f"=== Epoch {j+1}/{m} ===", end="\r", flush=True)
    vae_seq.train()
    for x_batch, _ in train_loader:
        x_batch = x_batch.view(x_batch.size(0), -1).to(device)
        optimizer.zero_grad()
        x_hat, z_mean, z_log_var = ...  ### TO BE COMPLETED ###
        loss = vae_loss(x_batch, x_hat, z_mean, z_log_var)
        loss.backward()
        optimizer.step()

    # Reconstruction at the end of epoch j
    vae_seq.eval()
    with torch.no_grad():
        x_test_decoded = vae_seq(x_test_sample)[0].view(n, 28, 28)

    imgs.append(...)   ### TO BE COMPLETED ###
    titles.append(...) ### TO BE COMPLETED ###

print("Training complete!")

sz = [(28, 28)] * len(imgs)
plot_images(imgs, sz, titles, n)

In [ ]:
# %load solutions/vae/training_visualization.py

## Classification of the latent variable

How are the different digits distributed over the latent space?

In order to study the compromise induced by the reconstruction / KL balance in the loss, we
introduce the following loss. We speak of a $\beta$-VAE when training a VAE with it
([Higgins et al., ICLR 2017](https://openreview.net/forum?id=Sy2fzU9gl)).

The two terms pull in opposite directions: the reconstruction term wants each image to have its
own well-separated place in the latent space, the KL term wants every $q(z\vert x)$ to look like
the prior, hence to overlap. $\beta$ says which one wins.

In [ ]:
def vae_loss_terms(x, x_hat, mean, log_var):
    """The two terms of the loss, returned separately, each summed over the batch.

    Same two quantities as in `vae_loss`; keeping them apart is what lets the training
    loop record them separately and lets us watch the compromise happen.
    """
    reconstruct_loss = F.binary_cross_entropy(x_hat, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reconstruct_loss, kl


def beta_vae_loss(x, x_hat, mean, log_var, beta=1):
    """Reconstruction + beta * KL. With beta = 1, this is exactly `vae_loss`."""
    reconstruct_loss, kl = vae_loss_terms(x, x_hat, mean, log_var)
    return reconstruct_loss + beta * kl

##### <i style="color:teal">**Todo:** Draw the scatter plot of the encoded digits, each point coloured according to its label</i>

Several values of $\beta$ can be considered. To help you, start by writing a `train_beta_vae`
function training a $\beta$-VAE for a given $\beta$.

In [ ]:
### TO BE COMPLETED ###

def train_beta_vae(vae,
                   beta = 1,
                   train_loader = train_loader,
                   test_loader = test_loader,
                   EPOCHS = 10,  #30
                   LEARNING_RATE = 1e-3,
                   device = device,
                   use_tqdm = True
                  ):
    """Train a beta-VAE and return it along with the history of the two terms of the loss.

    Record the reconstruction term and the KL term separately, per image, and store the
    KL **without** the factor beta: two runs with different values of beta are then
    comparable, which the total loss is not.

    Record in `kl_per_dim`, at each epoch, the contribution of each latent coordinate to
    the KL, measured on the validation set. It will be used in the last section of this
    part.
    """

    vae = check_device_model(vae, device=device)
    optimizer = optim.Adam(vae.parameters(), lr=LEARNING_RATE)

    history = {k: [] for k in ['loss', 'recon', 'kl', 'val_loss', 'val_recon', 'val_kl',
                               'kl_per_dim']}

    [...] ### TO BE COMPLETED ###

    return vae, history

In [ ]:
# %load solutions/vae/train_beta_vae.py

In [ ]:
# %load solutions/vae/plot_latents_1.py

In [ ]:
# %load solutions/vae/plot_latents_2.py

The scatter plots show where the images end up; the curves below show *why*. Since
`train_beta_vae` records the two terms separately, and records the KL without its factor
$\beta$, the three runs can be put on the same axes.

##### <i style="color:teal">**Todo:** Plot the reconstruction term and the KL term along the epochs, for the three values of $\beta$</i>

One panel per term, one curve per $\beta$. Then print the final value of each term, and of the
total loss, in a small table.

In [ ]:
### TO BE COMPLETED ###

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

[...] ### TO BE COMPLETED ###

plt.show()

In [ ]:
# %load solutions/vae/beta_curves.py

##### <i style="color:teal">**Question:** What do you observe as $\beta$ grows?</i>

**[Solution]**

<!--
With a small beta, the KL term barely counts: the encoder is free to spread the images wherever
it likes, the clusters are well separated and far apart, but the latent space no longer looks
like the prior, and sampling N(0, I) then lands in regions the decoder has never seen, so
generation degrades.

With a large beta, the opposite: every q(z|x) is pushed towards the prior, the clusters overlap
and, in the limit, the latent space stops carrying any information about the image. This is
posterior collapse.

beta = 1 is the ELBO, and it is the value for which the loss has a probabilistic meaning. The
other values trade likelihood for a latent space with other properties, disentanglement being
the one Higgins et al. were after.
-->

### How many latent coordinates are actually used? <small style="color:orangered">(to go further)</small>

The KL term is a sum over the coordinates of the latent space,
$$ KL\big(q(z\vert x)\,\Vert\,p(z)\big) \,=\, \sum_{j=1}^{d} KL_j \,, \qquad
   KL_j \,=\, \tfrac12\,\big[\sigma_{z,j}^2(x) + \mu_{z,j}^2(x) - 1 - \log\sigma_{z,j}^2(x)\big] \,. $$

Averaged over the images, $KL_j$ says how much information the encoder writes into coordinate
$j$. When it falls to zero, $q(z_j\vert x)$ equals the prior for *every* image: the encoder puts
nothing there and the decoder reads nothing from it. That coordinate has died, and the
phenomenon is known as **posterior collapse**.

With $d = 2$ there is little to see. We therefore rerun the same comparison with a latent space
of dimension 10, where the number of surviving coordinates becomes readable.

> This trains three more models. The number of epochs is kept low on purpose: the collapse
> shows up early, well before the reconstructions become good.

In [ ]:
LATENT_DIM_FAR = 10
EPOCHS_FAR = 10

vaes_10d = {}
histories_10d = {}

for beta in betas:
    print(f"=== beta={beta}, latent_dim={LATENT_DIM_FAR} ===")
    v = VAE_mlp(input_dim, intermediate_dim, LATENT_DIM_FAR).to(device)
    v, h = train_beta_vae(v, beta=beta, EPOCHS=EPOCHS_FAR, use_tqdm=False)
    vaes_10d[beta] = v
    histories_10d[beta] = h
    print('')

##### <i style="color:teal">**Todo:** Plot the contribution of each latent coordinate to the KL, for the three values of $\beta$</i>

One bar per coordinate, one group per $\beta$, coordinates sorted by decreasing contribution:
their numbering carries no meaning, only their number does. Then count, for each $\beta$, how
many coordinates stay above a small threshold.

`history['kl_per_dim']` holds that vector for each epoch, measured on the validation set.

In [ ]:
### TO BE COMPLETED ###

fig, ax = plt.subplots(figsize=(10, 5))

[...] ### TO BE COMPLETED ###

plt.show()

In [ ]:
# %load solutions/vae/kl_per_dimension.py

##### <i style="color:teal">**Question:** How does the number of surviving coordinates change with $\beta$, and what does the model lose when they die?</i>

**[Solution]**

<!--
The larger beta, the fewer coordinates survive. With beta = 10 only one or two carry anything,
and the total KL is small: the encoder has given up almost all the information it could have
written into z, because the loss charges it dearly. What the model loses is precisely the ability
to tell images apart: everything it drops from z has to be guessed by the decoder, which can only
answer with the average digit. This is why the reconstructions degrade.

With beta = 0.1 the opposite: nearly all ten coordinates are used and the total KL is large. The
latent space carries a lot, but it no longer resembles the prior, so drawing from N(0, I) lands
in regions the decoder was never trained on.

Note that the dead coordinates are not wasted parameters that could be pruned: which ones die is
an accident of the initialization, and the count is what matters, not the identity. Note also
that the collapse is not always something to fight: it is also how a VAE tells you that its
latent space is larger than the data needs.
-->

## New number generation

The generative nature of the VAE can be used to produce new data, here new digits.

##### <i style="color:teal">**Todo:** Generate a new image</i>

Draw a realization of the latent random variable $z$ from the prior, and pass it through the
decoder. Mind convention 3: that draw must be on the same `device` as the model.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/new_number.py

One image says little. The cell below draws forty independent samples of the prior and decodes
them all: this is what the model generates, misses included.

In [ ]:
# %load solutions/vae/prior_samples.py

The manifold associated with the latent space can also be visualized. The grid below is built
with the quantiles of the standard normal distribution rather than a regular grid: this way the
points explore the region of the latent space where the prior actually puts its mass.

In [ ]:
n_grid = 15  # figure with 15x15 panels
digit_size = 28

figure = np.zeros((digit_size * n_grid, digit_size * n_grid))
grid_x = norm.ppf(np.linspace(0.05, 0.95, n_grid))
grid_y = norm.ppf(np.linspace(0.05, 0.95, n_grid))

vae.eval()
with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xj in enumerate(grid_x):
            z_sample = torch.tensor([[xj, yi]], dtype=torch.float32, device=device)
            x_decoded = vae.decoder(z_sample)
            digit = x_decoded[0].cpu().numpy().reshape(digit_size, digit_size)

            figure[
                i * digit_size: (i+1) * digit_size,
                j * digit_size: (j+1) * digit_size
            ] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap="gray")
plt.axis("off")
plt.show()

## Interpolating in the latent space

Sampling the prior shows that the latent space is *populated*. Walking from one image to another
asks whether it is *continuous*.

We take two test images, encode them, and decode the points of the segment joining the two
codes. To keep the comparison fair we do it twice: with the VAE, and with a dense autoencoder of
the **same latent dimension**. Whatever differs then comes from the loss, not from the capacity.

> For the VAE we interpolate between the two means $\mu_z(x_a)$ and $\mu_z(x_b)$, and not between
> two draws of $z$: the question is where the encoder *places* each image, not where one sample
> happened to fall.
>
> Fair and spectacular do not go together here: at dimension 2 both models are so constrained
> that the two rows end up close. The answer below says what can, and cannot, be read on the
> figure.

The comparison needs an autoencoder of the same latent dimension as the VAE. The one below is
the improved autoencoder of the autoencoder lab (two dense layers each side, sigmoid on the
output, binary cross-entropy), reproduced here as provided code so that this notebook runs on
its own.

In [ ]:
# --- The improved autoencoder of the autoencoder lab, reproduced here ---

class DenseEncoder(nn.Module):
    """784 -> 128 -> n_latent, with a ReLU on the hidden layer only."""

    def __init__(self, n_input, n_latent):
        super().__init__()
        self.fc1 = nn.Linear(n_input, 128)
        self.fc2 = nn.Linear(128, n_latent)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class DenseDecoder(nn.Module):
    """n_latent -> 128 -> 784, ending with a sigmoid so the output lives in [0, 1]."""

    def __init__(self, n_latent, n_output):
        super().__init__()
        self.fc1 = nn.Linear(n_latent, 128)
        self.fc2 = nn.Linear(128, n_output)

    def forward(self, x):
        return torch.sigmoid(self.fc2(F.relu(self.fc1(x))))


class DenseAutoencoder(nn.Module):
    def __init__(self, n_input, n_latent):
        super().__init__()
        self.encoder = DenseEncoder(n_input, n_latent)
        self.decoder = DenseDecoder(n_latent, n_input)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_dense_autoencoder(autoencoder, EPOCHS=10, LEARNING_RATE=1e-3, device=device):
    """Plain reconstruction training, the loop of the autoencoder lab in short form."""
    autoencoder = check_device_model(autoencoder, device=device)
    optimizer = optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()

    for epoch in range(EPOCHS):
        autoencoder.train()
        train_loss = 0
        for inputs, _ in train_loader:
            inputs = inputs.view(inputs.size(0), -1).to(device)
            optimizer.zero_grad()
            loss = criterion(autoencoder(inputs), inputs)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss / len(train_loader.dataset):.4f}")

    return autoencoder

##### <i style="color:teal">**Todo:** Write an `interpolate` function and compare the two models on the same pair of images</i>

Two rows of `N_STEPS` images, the autoencoder on top and the VAE below. The function takes a
`variational` flag, since the two encoders do not return the same thing.

In [ ]:
N_STEPS = 10
idx_a, idx_b = 0, 1  # the two test images to travel between

# Flattened and sent to the device
x_a = test_dataset[idx_a][0].view(1, -1).to(device)
x_b = test_dataset[idx_b][0].view(1, -1).to(device)

# A dense autoencoder with the same latent dimension as the VAE
autoencoder_2d = DenseAutoencoder(input_dim, latent_dim)
autoencoder_2d = train_dense_autoencoder(autoencoder_2d, EPOCHS=10)

In [ ]:
### TO BE COMPLETED ###

def interpolate(model, x_a, x_b, n_steps=10, variational=False):
    "Decode along the segment joining the codes of two images."

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/vae/interpolation.py

##### <i style="color:teal">**Question:** What differs between the two rows, and why?</i>

**[Solution]**

<!--
This comparison is not conclusive, and it is more useful to say why than to read something into
it.

With a latent space of dimension **2**, both models are pushed well past what they can do. Neither
row is sharp, and depending on the run the autoencoder may not even return the two endpoint
images correctly: its row can end on a digit that is not the one it was asked to reach. The first
thing the figure shows is therefore that here the dimension of the latent space, and not the
loss, is the binding constraint.

The comparison the exercise was after is spoiled by the same constraint. At dimension 2 the
autoencoder's codes have nowhere to spread out: they are packed into a small region, the segment
joining two of them stays inside territory the decoder has already seen, and the walk stays
plausible for the wrong reason. The effect we wanted to see, a decoder producing nothing
meaningful between two codes, needs a much larger latent space, in which the codes form isolated
islands with empty space in between.

Running the top row again with the 32-dimensional autoencoder of the autoencoder lab is the way
to see the real difference, and it confirms that the dimension weighs as much as the loss.
-->

## Anomaly detection

In this section we see how variational autoencoders can be used for anomaly detection. We treat
the digit 9 as an outlier and build four datasets out of the training and test sets:

* training and test data without the 9s,
* training and test data with only the images of 9.

The VAE is then trained on the first family alone. Everything that follows rests on one idea: a
model that has never seen a 9 should reconstruct it worse than the digits it was trained on.

In [ ]:
BATCH_SIZE = 64
outlier_i = 9

In [ ]:
def make_dataset(x, y, label, target_class=outlier_i, batch_size=BATCH_SIZE, shuffle=False):
    """Build the dataset of the outliers (label=1) or of the regular digits (label=0).

    `label` is used twice here: to select the images, and as the target of the anomaly
    detection itself.

    The images are truncated to a whole number of batches rather than dropped by the
    loader: this way `len(loader.dataset)` counts exactly the images that are seen, which
    is what the training loop divides the loss by.
    """
    mask = (y == target_class) if label == 1 else (y != target_class)
    x_select = x[mask]
    n = x_select.shape[0] - x_select.shape[0] % batch_size  # ensure full batches
    x_select = x_select[:n]
    x_select = x_select.unsqueeze(1).float() / 255.0  # counterpart of the v2 pipeline above
    y_select = torch.full((n,), label, dtype=torch.long)
    dataset = TensorDataset(x_select, y_select)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataset, loader


# --- Raw data ---
x_train, y_train = train_dataset.data, train_dataset.targets
x_test, y_test = test_dataset.data, test_dataset.targets

# --- DataLoaders for anomaly detection ---
# Only the training loader is shuffled, as everywhere else in this lab
_, train_ad_loader              = make_dataset(x_train, y_train, label=0, shuffle=True)  # normal train
test_ad_dataset, test_ad_loader = make_dataset(x_test, y_test, label=0)                  # normal test
_, train_anomaly_loader                   = make_dataset(x_train, y_train, label=1, shuffle=True)  # anomaly train
test_anomaly_dataset, test_anomaly_loader = make_dataset(x_test, y_test, label=1)                  # anomaly test

test_ad_anomaly_dataset = ConcatDataset([test_ad_dataset, test_anomaly_dataset])
test_ad_anomaly_loader = DataLoader(test_ad_anomaly_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
EPOCHS = 30
vae = VAE_mlp(input_dim, intermediate_dim, latent_dim).to(device)
vae, _ = train_beta_vae(vae, beta=1,
                        train_loader=train_ad_loader, test_loader=test_ad_loader,
                        EPOCHS=EPOCHS, use_tqdm=False)

### Images decoded by the VAE

Let us now use our VAE on the test set of known digits (0 to 8) on the one hand, and on the
outliers (the 9s) on the other. The helper below is given: it passes a whole loader through the
model and returns the originals and their reconstructions, as `numpy` arrays.

In [ ]:
# %load solutions/vae/reconstruct.py

##### <i style="color:teal">**Todo:** Compare the reconstruction of regular and outlier data, for the VAE trained only on regular data</i>

* display sample images of the regular test set and their reconstructions;
* do the same for the outliers.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/reconstructions_comparison.py

### Anomaly detection using the latent representation

By performing a "regular vs outliers" clustering in the latent space, we may hope to detect the
anomalies.

##### <i style="color:teal">**Todo:** Compare the distribution of the encodings of regular and outlier data</i>

* display the scatter plot of the regular data and of the outliers on the same graph;
* can a clustering technique easily be applied?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/ad_detection_latents.py

### Distribution of the reconstruction error

To evaluate the detection of the images of $9$, let us look at the distribution of the mean
square error between the original image and its reconstruction,
$$ \mathrm{MSE}(x, \hat{x}) \,=\, \frac{1}{784} \sum_{k=1}^{784} \big(x_k - \hat{x}_k\big)^2 \,, $$
for three kinds of images:

* images of a known digit ($0$ to $8$),
* outlier images ($9$),
* completely random images.

The third family is a sanity check rather than a realistic case: if random noise were not
reconstructed much worse than everything else, the error would not be measuring anything.

##### <i style="color:teal">**Todo:** Compute the reconstruction error in each of the three situations</i>

In [ ]:
### TO BE COMPLETED ###

print("=== Reconstruction error over the different datasets ===")

# --- #
# Regular data
[...]
mse_regular = ...

# --- #
# Outliers data
[...]
mse_outliers = ...

# --- #
# Random data
[...]
mse_random = ...

In [ ]:
# %load solutions/vae/MSE_anomaly.py

##### <i style="color:teal">**Todo:** Build the histogram of the reconstruction errors for each of the three situations</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/vae/histograms_anomaly.py

##### <i style="color:teal">**Question:** What can be said about the reconstruction error in the different cases? Conclude on the performance of the VAE in detecting the anomalies.</i>

**[Solution]**

<!--
The random images are indeed reconstructed much worse than the rest, which confirms that the
error does measure something. But the histograms of the regular digits and of the 9s overlap
almost entirely: a threshold on the reconstruction error separates them badly.

Two reasons for this. A 9 is not far from a 4 or a 7, which the model has seen, so a decoder
with enough capacity reconstructs it passably. And a latent space of dimension 2 forces the
model to keep only the coarsest features, the ones the 9 shares with the other digits.

This is what the convolutional VAE at the end of the notebook is meant to improve.
-->

The performance of such a classifier can also be assessed with a ROC curve.

The score is the reconstruction error itself, and the positive class is the outlier: a large
error is expected to mean an anomaly. That direction is chosen beforehand, from what the model
is supposed to do, and not by looking at which of the two gives the better AUC. An AUC below
$0.5$ is a result, not something to be flipped.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
mse_score = np.concatenate([mse_regular, mse_outliers], 0)
true_label = [0]*mse_regular.shape[0] + [1]*mse_outliers.shape[0]

fpr, tpr, thresholds = roc_curve(true_label, mse_score)
auc_score = roc_auc_score(true_label, mse_score)

fig, ax = plt.subplots(1, 1, figsize=(9, 5))

ax.plot(fpr, tpr, 'c.-', label='ROC curve {:2.2f}'.format(auc_score))
ax.plot(fpr, fpr, 'k-', label='Random guessing')

ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
plt.legend()
plt.show()

The result is clearly not very conclusive. Before blaming the architecture, let us check whether
we are asking the right question of the model.

### Three scores rather than one

The reconstruction error answers one question: *can the model redraw this image?* It says nothing
about where the code of that image landed. The KL term answers the opposite question: *is this
code where the prior expects codes to be?*, without caring how the reconstruction turned out. A
$9$ that happens to look like a $4$ scores as perfectly normal on the first and suspicious on the
second.

Their sum is $-\mathrm{ELBO}$, which estimates $-\log p(x)$: *how likely is this image under my
model*, which is, stated properly, the anomaly detection question itself.

> **Principled is not the same as effective.** The ELBO is a lower bound whose gap varies from
> one point to the next, and deep generative models are known to assign a *higher* likelihood to
> some out-of-distribution data than to their own training data
> ([Nalisnick et al., ICLR 2019](https://openreview.net/forum?id=H1xwNhCcYm)). Add to this the
> caveat on the reconstruction term stated when the loss was defined: with grey levels and a
> plain binary cross-entropy, what we compute is an ELBO only up to a term we ignore. So the
> three scores are worth comparing, not ranking in advance.

##### <i style="color:teal">**Todo:** Compute the three scores per image and put their three ROC curves on the same axes</i>

A single pass over each loader is enough for the three. Careful to orient them the same way: for
all three, the larger the score, the more anomalous the image.

In [ ]:
### TO BE COMPLETED ###

def vae_scores(model, loader, device=device):
    "MSE, KL and -ELBO for each image of `loader`."

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/vae/anomaly_scores.py

##### <i style="color:teal">**Question:** Which of the three works best here? Was it the expected one?</i>

**[Solution]**

<!--
The MSE comes out a little above $0.5$, the $-\mathrm{ELBO}$ barely above it, and the KL far
*below* it: two runs of this notebook gave $0.30$ and $0.22$ for the KL, against $0.56$ and
$0.55$ for the MSE. The exact values move with the seed; the pattern does not, and it is the
pattern that matters. Two things are worth reading here, and neither of them is the ranking.

The KL sits far *below* 0.5, and that is not noise: it means the 9s land **closer** to the prior
than the digits the model was trained on. The reason is worth stating. Faced with an image it
has never seen, the encoder has no confident code to produce, and falls back on something close
to $\mu = 0$, $\sigma = 1$, that is, on the prior itself. So for this model the anomaly signal
carried by the KL is a *small* KL, not a large one. This is exactly the situation described in
the remark above the first ROC curve: an AUC below $0.5$ is a result to be read, not a sign to
be flipped. What would make $-KL$ a legitimate score is the argument just given about the
encoder falling back on the prior, stated before looking at the labels, not the AUC itself.

It also explains why $-\mathrm{ELBO}$ does worse than the MSE alone. Here its two terms point in
opposite directions (the reconstruction term weakly informative the expected way, the KL term
strongly informative the other way), and adding them cancels most of both. The best-founded
score is not the best score, and not by accident: an estimate of $-\log p(x)$ is only as good as
the bound behind it, and the work cited above shows that likelihood itself is an unreliable
out-of-distribution detector.

None of the three is usable as it stands, which is the honest conclusion of this section.
-->

Neither of the three scores saves the day: the architecture is the next lever.

## Convolutional variational autoencoders <small style="color:orangered">(to go further)</small>

We have seen how to build a VAE and how to use it to generate new images and detect anomalies.
The VAEs used so far only involve dense layers (MLP), which may explain their weakness on the
anomaly detection task.

##### <i style="color:teal">**Todo:** Use convolutional layers to build a convolutional VAE, and test it on the two applications (generating images and detecting anomalies)</i>

The convolutional autoencoder of the autoencoder lab gives the encoder and the decoder; what remains is to
put the two heads `z_mean` and `z_log_var` on top of the convolutional base, which means
flattening its output first. The loss and the training loop do not change.